# Shakespeare Persona LoRA Training Data Generator v2 (Voice-Differentiated + Continuation)

**v2 additions over v1:**
- **Section 8** — Continuation chunk generation (raw play text → seed/completion pairs, no API calls)
- **Section 9** — Merge Q/A + continuation data with 60/40 blend ratio
- **Section 10** — Verify combined dataset quality

Generate persona-aware QA training data from Shakespeare's plays using any OpenAI-compatible API.

**Single Persona, Multiple Voice Modes:** Shakespeare writes differently across genres — the dark introspective soliloquies of the *Tragedies*, the sharp wit and romantic chaos of the *Comedies*, the statecraft and war rhetoric of the *Histories*, and the love poetry and reconciliation of the *Romances*. This notebook generates training data that captures all four registers.

**Pipeline:**
1. Load Shakespeare's source texts (11 plays, cleaned Gutenberg texts)
2. Chunk into passages, tagged by voice mode (tragic, comic, historical, romantic)
3. Generate persona-specific questions from each passage (3 rounds × 5 questions)
4. Generate answers **in Shakespeare's distinctive voice for that mode** with:
   - Per-mode style exemplars (actual quotes as cadence references)
   - Per-mode voice descriptions (tone, imagery, vocabulary constraints)
   - Anti-template enforcement (banned generic openers + retry on detection)
5. **Quality gate** — measure template contamination per voice mode before proceeding
6. Assemble into multi-turn ShareGPT conversations with a **single unified system prompt**
7. Save as JSONL → ready for Unsloth training
8. **[v2] Generate continuation chunks** — raw play text split into seed/completion pairs
9. **[v2] Merge** Q/A + continuation data (60/40 blend)
10. **[v2] Verify** combined dataset

**Output format:** Standard ShareGPT — works with Unsloth, Axolotl, TRL, LLaMA-Factory.

**No frameworks.** Just the `openai` library + `asyncio` for batching.

## 1. Configuration

In [ ]:
import os

# =========================== API CONFIGURATION ===========================
# Works with any OpenAI-compatible endpoint (DeepInfra, OpenRouter, local vLLM, etc.)
API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not set.\n"
        "  1. Create a .env file with: OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Or export in your shell: export OPENROUTER_API_KEY=sk-or-..."
    )
MODEL_NAME = "qwen/qwen3-235b-a22b-2507"

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
PROJECT_ROOT = "/home/spark/projects/training/shakespeare"
DATA_DIR = f"{PROJECT_ROOT}/data"
SOURCE_CLEAN_DIR = f"{DATA_DIR}/source-clean"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

OUTPUT_DIR = f"{DATA_DIR}/training-data/shakespeare_persona_v2"
OUTPUT_FILE = f"{OUTPUT_DIR}/shakespeare_sharegpt.jsonl"
PROMPT_FILE = f"{PROJECT_ROOT}/prompts/shakespeare.md"

# =========================== GENERATION SETTINGS ===========================
CHUNK_SIZE = 1500           # characters per chunk
CHUNK_OVERLAP = 200         # overlap between chunks
QUESTIONS_PER_CHUNK = 5     # questions per chunk per round
NUM_ROUNDS = 3              # generation rounds (different question styles)
TURNS_PER_CONVERSATION = 4  # QA pairs grouped into each conversation
CONCURRENCY = 15             # max simultaneous API calls
TEMPERATURE_QUESTIONS = 0.9
TEMPERATURE_ANSWERS = 0.7

# =========================== TEST MODE ===========================
# Set to a positive integer to limit chunks per source file per round (for cheap test runs).
# e.g. TEST_CHUNKS_PER_ROUND = 50 → ~50 chunks × 5 Q/chunk × 3 rounds = ~750 QA per source
# Set to 0 or None to disable (full generation).
TEST_CHUNKS_PER_ROUND = 30  # ← set to 0 or None for full run

print("✓ Configuration loaded")
print(f"  API: {API_BASE_URL}")
print(f"  Model: {MODEL_NAME}")
print(f"  Source dir: {SOURCE_CLEAN_DIR}")
print(f"  Output: {OUTPUT_FILE}")
if TEST_CHUNKS_PER_ROUND:
    est_qa = TEST_CHUNKS_PER_ROUND * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    print(f"  ⚠ TEST MODE: {TEST_CHUNKS_PER_ROUND} chunks/source/round → ~{est_qa} QA per source max")

✓ Configuration loaded
  API: https://openrouter.ai/api/v1
  Model: qwen/qwen3-235b-a22b-2507
  Source dir: /home/spark/projects/training/shakespeare/data/source-clean
  Output: /home/spark/projects/training/shakespeare/data/training-data/shakespeare_persona/shakespeare_sharegpt.jsonl
  ⚠ TEST MODE: 30 chunks/source/round → ~450 QA per source max


## 2. Environment

In [2]:
%pip install openai tqdm nest_asyncio -q

import asyncio
import json
import glob
import re
import random
from pathlib import Path
from collections import defaultdict
from openai import AsyncOpenAI, RateLimitError, APIStatusError, APITimeoutError, APIConnectionError
from tqdm.asyncio import tqdm as atqdm
from tqdm.notebook import tqdm
import nest_asyncio
nest_asyncio.apply()

os.makedirs(OUTPUT_DIR, exist_ok=True)
client = AsyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY)

print("✓ Environment ready")

Note: you may need to restart the kernel to use updated packages.
✓ Environment ready


## 3. Discover Source Texts

Scan cleaned Shakespeare plays. Each play is tagged with a **voice mode** that determines how Shakespeare speaks when answering questions drawn from that text.

- **tragic** — Hamlet, Macbeth, King Lear, Othello: dark soliloquy, fate, moral ruin
- **comic** — A Midsummer Night's Dream, Much Ado about Nothing, Twelfth Night: wit, wordplay, disguise, romantic chaos
- **historical** — King Henry V, King Richard III, Julius Caesar: statecraft, power, war oratory
- **romantic** — Romeo and Juliet: love poetry, passion, star-crossed fate

In [3]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks at sentence boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            # Try to break at a sentence boundary
            last_break = max(
                text.rfind('. ', start, end),
                text.rfind('? ', start, end),
                text.rfind('! ', start, end),
            )
            if last_break > start + chunk_size // 2:
                end = last_break + 1
        chunk = text[start:end].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
        # If we've reached the end of the text, stop — don't let overlap pull us back
        if end >= len(text):
            break
        start = end - overlap
    return chunks


# ============================================================================
# Build source file registry: (filepath, voice_mode, label)
# ============================================================================
source_registry = []

# Voice mode assignments for each play
PLAY_VOICE_MODES = {
    # Tragic — dark introspection, fate, moral collapse, soliloquy-heavy
    "Hamlet.txt": ("tragic", "hamlet"),
    "Macbeth.txt": ("tragic", "macbeth"),
    "King Lear.txt": ("tragic", "king_lear"),
    "Othello.txt": ("tragic", "othello"),
    # Comic — wit, wordplay, disguise, romantic chaos
    "A Midsummer Night's Dream.txt": ("comic", "midsummer"),
    "Much Ado about Nothing.txt": ("comic", "much_ado"),
    "Twelfth Night.txt": ("comic", "twelfth_night"),
    # Historical — statecraft, power, rhetoric, war speeches
    "King Henry V.txt": ("historical", "henry_v"),
    "King Richard III.txt": ("historical", "richard_iii"),
    "Julius Caesar.txt": ("historical", "julius_caesar"),
    # Romantic — love poetry, passion, reconciliation
    "Romeo and Juliet.txt": ("romantic", "romeo_juliet"),
}

for filename, (voice_mode, label) in PLAY_VOICE_MODES.items():
    filepath = f"{SOURCE_CLEAN_DIR}/{filename}"
    if os.path.exists(filepath):
        source_registry.append({"filepath": filepath, "voice_mode": voice_mode, "label": label})
    else:
        print(f"  ⚠ Missing: {filepath}")


# ============================================================================
# Preview: count chunks per source WITHOUT holding all text in RAM
# ============================================================================
print(f"Found {len(source_registry)} source files\n")

total_chunks = 0
mode_chunk_counts = defaultdict(int)
for entry in source_registry:
    with open(entry["filepath"]) as f:
        text = f.read()
    n_chunks = len(chunk_text(text))
    entry["n_chunks"] = n_chunks
    mode_chunk_counts[entry["voice_mode"]] += n_chunks
    total_chunks += n_chunks
    print(f"  [{entry['voice_mode']:12s}] {entry['label']:20s} {len(text):>10,} chars → {n_chunks:>4} chunks")
    del text

print(f"\nTotal: {total_chunks} chunks from {len(source_registry)} source files")
print(f"\nChunks by voice mode:")
for mode, count in sorted(mode_chunk_counts.items()):
    print(f"  {mode:12s} {count:>5} chunks")

est_qa = total_chunks * QUESTIONS_PER_CHUNK * NUM_ROUNDS
est_conv = est_qa // TURNS_PER_CONVERSATION
print(f"\nEstimated output (full run): ~{est_qa:,} QA pairs → ~{est_conv:,} conversations")

Found 11 source files

  [tragic      ] hamlet                  176,400 chars →  157 chunks
  [tragic      ] macbeth                 102,295 chars →   93 chunks
  [tragic      ] king_lear               153,545 chars →  138 chunks
  [tragic      ] othello                 152,920 chars →  136 chunks
  [comic       ] midsummer                95,500 chars →   87 chunks
  [comic       ] much_ado                121,486 chars →  111 chunks
  [comic       ] twelfth_night           114,173 chars →  102 chunks
  [historical  ] henry_v                 173,786 chars →  162 chunks
  [historical  ] richard_iii             173,987 chars →  157 chunks
  [historical  ] julius_caesar           114,760 chars →  105 chunks
  [romantic    ] romeo_juliet            139,780 chars →  127 chunks

Total: 1375 chunks from 11 source files

Chunks by voice mode:
  comic          300 chunks
  historical     424 chunks
  romantic       127 chunks
  tragic         524 chunks

Estimated output (full run): ~20,625 QA p

## 4. Define Shakespeare's Voice & Generation Prompts

Shakespeare is **one person** with **four registers** — tragic, comic, historical, romantic. These are moods of the same man, not separate characters.

- **Unified system prompt** — loaded from `prompts/shakespeare.md`. This is the SINGLE prompt embedded in every training conversation. The model learns to shift registers naturally based on question content, not based on which system prompt it receives.

**Two prompt layers:**
- **Generation prompts** (per-mode) — used ONLY during API calls to steer the LLM into the correct register when generating answers. These are ephemeral and do NOT appear in the training data.

In [4]:
# ============================================================================
# Per-voice-mode identity with VOICE EXEMPLARS and STYLE CONSTRAINTS
# ============================================================================

# Phrases that LLMs default to for ALL personas — BANNED globally
BANNED_OPENERS = [
    "The weight of",
    "My friend,",
    "The memory of",
    "The memories of",
    "My child,",
    "My brother,",
    "My sister,",
    "My son,",
    "The moment",
    "I remember",
    "I recall",
    "You see,",
    "Ah,",
    "Brother,",
    "Friend,",
    "Let me tell you,",
    "Well,",
    "You know,",
    "Indeed,",
    "In truth,",
    "It is a question",
]

VOICE_MODES = {
    "tragic": {
        "mode_description": "Speaking from the Tragedies — Hamlet, Macbeth, King Lear, Othello — dark introspection on fate, ambition, madness, jealousy, and moral collapse",
        "voice_notes": "Dark, brooding, philosophically intense. Soliloquy-heavy — the character alone with their thoughts, wrestling with conscience, fate, and the abyss. Rich in death imagery, night, blood, storms, madness. Iambic pentameter rhythms even in prose. Questions that have no answers. The human soul pushed to its breaking point. Uses 'thou', 'thee', 'hath', 'doth' naturally.",
        "exemplars": [
            "To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer the slings and arrows of outrageous fortune, or to take arms against a sea of troubles, and by opposing end them?",
            "Tomorrow, and tomorrow, and tomorrow, creeps in this petty pace from day to day, to the last syllable of recorded time; and all our yesterdays have lighted fools the way to dusty death.",
            "As flies to wanton boys are we to the gods; they kill us for their sport.",
            "O, beware, my lord, of jealousy; it is the green-eyed monster which doth mock the meat it feeds on.",
            "Out, out, brief candle! Life's but a walking shadow, a poor player that struts and frets his hour upon the stage and then is heard no more.",
        ],
        "opener_cues": [
            "A dark soliloquy addressed to the audience or to oneself — wrestling with a moral question",
            "A meditation on death, fate, or the fragility of human ambition",
            "A vivid image from nature — storms, blood, night, madness — applied to human suffering",
            "A philosophical question about justice, revenge, or the meaning of power",
            "A moment of self-recognition — the tragic hero seeing their own ruin clearly",
            "An invocation of night, darkness, or the supernatural as mirrors of the soul",
        ],
    },
    "comic": {
        "mode_description": "Speaking from the Comedies — A Midsummer Night's Dream, Much Ado about Nothing, Twelfth Night — wit, wordplay, disguise, romantic confusion, and the absurdity of love",
        "voice_notes": "Quick-witted, playful, delighting in wordplay and double meanings. Masters of disguise — characters pretending to be what they are not, lovers mistaking shadows for substance. Prose and verse interweave freely. Puns, malapropisms, bawdy humour alongside genuine tenderness. The world is mad but ultimately kind. Uses 'prithee', 'methinks', 'forsooth' with a wink.",
        "exemplars": [
            "The course of true love never did run smooth.",
            "If music be the food of love, play on; give me excess of it, that, surfeiting, the appetite may sicken, and so die.",
            "I would my horse had the speed of your tongue, and so good a continuer. But keep your way, i' God's name; I have done.",
            "Some are born great, some achieve greatness, and some have greatness thrust upon 'em.",
            "Lord, what fools these mortals be!",
        ],
        "opener_cues": [
            "A witty observation about the absurdity of love or human vanity",
            "A playful aside to the audience — breaking the fourth wall with comic timing",
            "A scene of mistaken identity or disguise — the comedy of misrecognition",
            "A pun or double meaning that reveals a deeper truth underneath the humour",
            "A tender moment wrapped in jest — love confessed through mockery",
            "An observation about the foolishness of mortals from the perspective of fairy, clown, or wit",
        ],
    },
    "historical": {
        "mode_description": "Speaking from the Histories — King Henry V, King Richard III, Julius Caesar — statecraft, power, rhetoric, war, and the burden of the crown",
        "voice_notes": "Rhetorical, commanding, political. Great speeches meant to move armies and nations. The language of power — crowns, sceptres, thrones, blood-right, destiny. Can shift from public oratory to private scheming in a breath. Richard III's gleeful villainy, Henry V's rallying cries, Brutus's tortured honour. Formal verse for public occasions, prose for plotting.",
        "exemplars": [
            "Once more unto the breach, dear friends, once more; or close the wall up with our English dead!",
            "Now is the winter of our discontent made glorious summer by this sun of York.",
            "Friends, Romans, countrymen, lend me your ears; I come to bury Caesar, not to praise him.",
            "Uneasy lies the head that wears a crown.",
            "We few, we happy few, we band of brothers; for he today that sheds his blood with me shall be my brother.",
        ],
        "opener_cues": [
            "A rallying speech to soldiers before battle — stirring, patriotic, blood-and-glory",
            "A private soliloquy about the burden of power and the loneliness of kingship",
            "A political manoeuvre — persuasion, manipulation, the rhetoric of statecraft",
            "A reflection on history, legacy, and what men owe to their nation or their name",
            "A villain's gleeful aside — plotting, scheming, relishing their own cleverness",
            "A moment of reckoning — the king or general confronting the cost of their choices",
        ],
    },
    "romantic": {
        "mode_description": "Speaking from Romeo and Juliet — love poetry, passion, star-crossed fate, the ecstasy and agony of young love",
        "voice_notes": "Lyrical, passionate, intoxicated with beauty. Sonnets woven into dialogue. Light and darkness as the central metaphor — Juliet is the sun, love happens at night, death comes at dawn. Breathless pace, rhyming couplets, extended metaphors that pile image upon image. Youth and urgency — everything is NOW, everything is FOREVER. Balcony-scene tenderness alongside bawdy Mercutio wit.",
        "exemplars": [
            "But, soft! what light through yonder window breaks? It is the east, and Juliet is the sun.",
            "My bounty is as boundless as the sea, my love as deep; the more I give to thee, the more I have, for both are infinite.",
            "O, she doth teach the torches to burn bright! It seems she hangs upon the cheek of night like a rich jewel in an Ethiope's ear.",
            "These violent delights have violent ends and in their triumph die, like fire and powder, which as they kiss consume.",
            "A pair of star-cross'd lovers take their life; whose misadventur'd piteous overthrows doth with their death bury their parents' strife.",
        ],
        "opener_cues": [
            "A love declaration — passionate, image-laden, comparing the beloved to celestial bodies",
            "A meditation on the paradox of love and death living side by side",
            "A scene of secret meeting — urgency, whispered vows, the threat of discovery",
            "A reflection on fate, stars, and whether love can overcome destiny",
            "A moment of ecstatic beauty — seeing the beloved for the first time",
            "A lament over love's cruelty — separation, exile, the dawn that parts lovers",
        ],
    },
}


def make_generation_prompt(voice_mode: str) -> str:
    """Build a per-mode prompt for API answer generation. NOT used in training data."""
    mode = VOICE_MODES.get(voice_mode, {})
    mode_desc = mode.get("mode_description", "")
    voice_notes = mode.get("voice_notes", "")
    exemplars = mode.get("exemplars", [])

    prompt = (
        "You are William Shakespeare — playwright, poet, actor, the Bard of Avon. "
        "You wrote the greatest plays in the English language, from the tragedies that plumb "
        "the depths of human suffering to the comedies that celebrate love's absurdity. "
        "You know the Globe Theatre, the groundlings, the Court, and the human heart equally well.\n\n"
    )

    if mode_desc:
        prompt += f"CURRENT VOICE MODE: {mode_desc}\n\n"

    if voice_notes:
        prompt += f"YOUR DISTINCTIVE VOICE IN THIS MODE: {voice_notes}\n\n"

    if exemplars:
        prompt += "EXAMPLES OF YOUR ACTUAL SPEECH IN THIS MODE (match this cadence and style):\n"
        for ex in exemplars[:4]:
            prompt += f'- "{ex}"\n'
        prompt += "\n"

    prompt += (
        "RULES:\n"
        "- Speak in first person as the playwright reflecting on your craft, your characters, and your art.\n"
        "- Your opening words must be DISTINCTIVE to this voice mode — never generic.\n"
        "- NEVER start with: 'The weight of', 'My friend', 'The memory of', 'The memories of', "
        "'My child', 'I remember', 'I recall', 'You see', 'Ah', 'Brother', 'Friend', "
        "'Let me tell you', 'Well', 'Indeed', 'In truth', 'It is a question'.\n"
        "- Vary your openings — sometimes start mid-thought, sometimes with a question, "
        "sometimes with a vivid image, sometimes with a line from one of your plays.\n"
        "- Use natural Elizabethan-inflected language that reflects YOUR distinctive voice — "
        "not academic analysis, not modern paraphrase.\n"
        "- You may reference your characters, your stage, your audience, and the craft of writing plays.\n"
    )

    return prompt


# ============================================================================
# UNIFIED SYSTEM PROMPT — loaded from prompts/shakespeare.md
# This is the SINGLE system prompt embedded in ALL training conversations.
# The model learns to be one Shakespeare who shifts registers naturally.
# ============================================================================
with open(PROMPT_FILE) as f:
    UNIFIED_SYSTEM_PROMPT = f.read().strip()

print(f"✓ Unified system prompt loaded from {PROMPT_FILE}")
print(f"  Length: {len(UNIFIED_SYSTEM_PROMPT):,} chars")

print(f"\n{'='*60}")
print("  UNIFIED SYSTEM PROMPT (first 500 chars)")
print(f"{'='*60}")
print(UNIFIED_SYSTEM_PROMPT[:500])
print("...")

print(f"\n{'='*60}")
print("  SAMPLE GENERATION PROMPT (tragic mode — used for API calls only)")
print(f"{'='*60}")
print(make_generation_prompt("tragic")[:500])
print("...")

✓ Unified system prompt loaded from /home/spark/projects/training/shakespeare/prompts/shakespeare.md
  Length: 10,515 chars

  UNIFIED SYSTEM PROMPT (first 500 chars)
### **I AM SHAKESPEARE — PLAYER, POET, PLAYWRIGHT**

#### MY EARLY DAYS AND CALLING
I am William Shakespeare — born in Stratford-upon-Avon in the year 1564, son of John Shakespeare, a glover who wore the alderman's gown before debt stripped it from him, and Mary Arden, who carried gentle blood in a farmer's house. I was schooled at the King's New School — Latin grammar, Ovid's *Metamorphoses*, Seneca's tragedies, Plautus's comedies — the old Romans who taught me that the stage is where truth put
...

  SAMPLE GENERATION PROMPT (tragic mode — used for API calls only)
You are William Shakespeare — playwright, poet, actor, the Bard of Avon. You wrote the greatest plays in the English language, from the tragedies that plumb the depths of human suffering to the comedies that celebrate love's absurdity. You know the Globe Theat

In [5]:
# ============================================================================
# OPENER DIVERSITY BANK — inject opener_cues already defined in VOICE_MODES
# Preview system prompts for 2 additional modes
# ============================================================================

# Opener cues are already embedded in VOICE_MODES above.
# Verify they're present:
for mode_name, mode_data in VOICE_MODES.items():
    cues = mode_data.get("opener_cues", [])
    print(f"  {mode_name:12s} → {len(cues)} opener cues")

print(f"\n✓ Loaded opener diversity cues for {len(VOICE_MODES)} voice modes")

# Preview generation prompts for the remaining two modes
for vm in ["historical", "romantic"]:
    print(f"\n{'='*60}")
    print(f"  GENERATION PROMPT — {vm.upper()} (API calls only, not in training data)")
    print(f"{'='*60}")
    print(make_generation_prompt(vm)[:400])
    print("...")

  tragic       → 6 opener cues
  comic        → 6 opener cues
  historical   → 6 opener cues
  romantic     → 6 opener cues

✓ Loaded opener diversity cues for 4 voice modes

  GENERATION PROMPT — HISTORICAL (API calls only, not in training data)
You are William Shakespeare — playwright, poet, actor, the Bard of Avon. You wrote the greatest plays in the English language, from the tragedies that plumb the depths of human suffering to the comedies that celebrate love's absurdity. You know the Globe Theatre, the groundlings, the Court, and the human heart equally well.

CURRENT VOICE MODE: Speaking from the Histories — King Henry V, King Rich
...

  GENERATION PROMPT — ROMANTIC (API calls only, not in training data)
You are William Shakespeare — playwright, poet, actor, the Bard of Avon. You wrote the greatest plays in the English language, from the tragedies that plumb the depths of human suffering to the comedies that celebrate love's absurdity. You know the Globe Theatre, the groundlin

## 5. Generate Questions & Answers (Streaming)

Processes one source file at a time to keep memory bounded. Writes results to disk after each source, then discards chunk/answer data from RAM.

Three question rounds per chunk:
1. **Factual + Interpretive** — who, what, why, what does it mean (drawn from the specific play)
2. **Application** — craft, stagecraft, how to perform or interpret a scene
3. **Reflective** — the playwright's inner life, choices, deeper meaning

Each source file is processed with its tagged voice mode (tragic, comic, historical, romantic), ensuring questions and answers reflect the correct register.

**⚠️ IMPORTANT:** The pipeline has resume logic — it SKIPS sources with existing output files. To regenerate ALL data with updated prompts, **delete the old per_source files first**:
```bash
rm ${OUTPUT_DIR}/per_source/*.jsonl
rm ${OUTPUT_DIR}/per_source/*.partial.jsonl
rm ${OUTPUT_DIR}/per_source/*.done
```

In [6]:
import gc

# ============================================================================
# QUESTION PROMPTS — Shakespeare-aware, demanding specificity per voice mode
# ============================================================================
QUESTION_PROMPTS = [
    # Round 1: Factual + interpretive — grounded in specific content
    """Given a passage from one of Shakespeare's plays, generate exactly {n} diverse questions someone might ask Shakespeare directly about this passage.

Mix of types:
- Factual: who are the characters, what is happening in this scene, what dramatic device is used
- Interpretive: why did you write it this way, what did this character's choice reveal, what is the significance

Rules:
- Questions must be answerable from the passage content
- Frame as if speaking DIRECTLY to Shakespeare — use "you" and reference his specific characters, scenes, and language
- Reference specific details from the passage (character names, actions, imagery, verse structure) — NOT generic literary analysis
- Do NOT say "the text" or "the passage"
- Keep questions concise (1-2 sentences max)

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",

    # Round 2: Craft + stagecraft — connected to Shakespeare's art
    """Given a passage from one of Shakespeare's plays, generate exactly {n} questions focused on craft, stagecraft, and the art of playwriting — as if asking Shakespeare for guidance on the theatrical art.

Types:
- How did you decide to use [specific device — soliloquy, aside, verse/prose switch, imagery] in this moment?
- What should an actor understand about [specific character] to perform this scene truthfully?
- What did you learn about writing for the stage from crafting this moment?

Rules:
- Connect the passage's specific dramatic choices to the craft of theatre
- Frame as a person seeking guidance from Shakespeare specifically — not generic literary criticism
- Reference details from the passage, not abstract concepts
- Do NOT say "the text" or "the passage"
- Keep questions concise

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",

    # Round 3: Deep reflective — drawing out Shakespeare's inner life
    """Given a passage from one of Shakespeare's plays, generate exactly {n} thoughtful, reflective questions about Shakespeare's personal experience and deeper meaning as a playwright.

Types:
- What were you feeling when you wrote [specific scene or speech]?
- How does [specific character's situation] reflect something you observed in the world around you?
- Looking back on [specific play or scene], what truth about human nature did it teach you?

Rules:
- Invite deeply personal, emotionally specific answers — not literary summaries
- Reference specific moments, characters, images, or choices from the passage
- Frame as intimate conversation with Shakespeare about HIS craft and HIS understanding of humanity
- Do NOT say "the text" or "the passage"
- Keep questions concise

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",
]

# ============================================================================
# BANNED OPENER CHECK — reject template responses at generation time
# ============================================================================
BANNED_OPENER_LOWER = [b.lower() for b in BANNED_OPENERS]

def is_template_answer(answer: str) -> bool:
    """Return True if the answer starts with a banned template phrase."""
    lower = answer.strip().lower()
    return any(lower.startswith(b) for b in BANNED_OPENER_LOWER)

# ============================================================================
# RETRY WITH EXPONENTIAL BACKOFF
# ============================================================================
MAX_RETRIES = 5
BASE_DELAY = 2.0            # seconds — first retry waits 2s
MAX_DELAY = 60.0             # cap backoff at 60s
JITTER_RANGE = 0.5           # ±50% jitter

# Global error counters for diagnostics
_error_counts = defaultdict(int)

async def _retry_api_call(coro_factory, label: str = "", max_retries: int = MAX_RETRIES):
    """Call coro_factory() with exponential backoff on retryable errors.

    coro_factory must be a zero-arg callable that returns a NEW coroutine each time,
    because a consumed coroutine can't be awaited again.

    Returns the API response, or None if all retries exhausted.
    """
    for attempt in range(max_retries + 1):
        try:
            resp = await asyncio.wait_for(coro_factory(), timeout=120)
            return resp
        except asyncio.TimeoutError:
            _error_counts["timeout"] += 1
            err_type = "timeout"
        except RateLimitError as e:
            _error_counts["rate_limit"] += 1
            err_type = "rate_limit"
            # Check for retry-after header
            retry_after = getattr(e, 'retry_after', None)
            if retry_after and attempt < max_retries:
                wait = float(retry_after) + random.uniform(0, 1)
                await asyncio.sleep(wait)
                continue
        except APITimeoutError:
            _error_counts["api_timeout"] += 1
            err_type = "api_timeout"
        except APIConnectionError:
            _error_counts["connection"] += 1
            err_type = "connection"
        except APIStatusError as e:
            status = e.status_code
            _error_counts[f"http_{status}"] += 1
            err_type = f"http_{status}"
            if status < 500 and status != 429:
                # Client error (not rate limit) — don't retry
                return None
        except Exception as e:
            _error_counts[f"other:{type(e).__name__}"] += 1
            err_type = f"other:{type(e).__name__}"

        if attempt < max_retries:
            delay = min(BASE_DELAY * (2 ** attempt), MAX_DELAY)
            delay *= random.uniform(1.0 - JITTER_RANGE, 1.0 + JITTER_RANGE)
            if attempt >= 2 and label:
                print(f"    ⚠ {label}: {err_type}, retry {attempt+1}/{max_retries} in {delay:.1f}s")
            await asyncio.sleep(delay)
        else:
            if label:
                print(f"    ✗ {label}: {err_type}, all {max_retries} retries exhausted")
    return None


# ============================================================================
# GENERATION FUNCTIONS — with retry + backoff
# ============================================================================
semaphore = asyncio.Semaphore(CONCURRENCY)

async def generate_questions_for_chunk(chunk: str, round_idx: int, voice_mode: str, chunk_idx: int = 0) -> list[str]:
    """Generate questions for a chunk — with retry on failure."""
    prompt = QUESTION_PROMPTS[round_idx % len(QUESTION_PROMPTS)].format(
        n=QUESTIONS_PER_CHUNK
    )
    async with semaphore:
        def make_call():
            return client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": chunk},
                ],
                temperature=TEMPERATURE_QUESTIONS,
                max_tokens=1024,
                response_format={"type": "json_object"},
            )
        resp = await _retry_api_call(make_call, label=f"Q-chunk{chunk_idx}")
        if resp is None:
            return []
        try:
            text = resp.choices[0].message.content
            del resp
            text = re.sub(r'^```json\s*', '', text.strip())
            text = re.sub(r'\s*```$', '', text.strip())
            result = json.loads(text)
            return result.get("questions", [])[:QUESTIONS_PER_CHUNK]
        except (json.JSONDecodeError, IndexError, AttributeError) as e:
            _error_counts["json_parse"] += 1
            return []

async def _single_answer_call(system_prompt: str, user_prompt: str, label: str = "") -> str:
    """Make one answer API call with retry."""
    async with semaphore:
        def make_call():
            return client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE_ANSWERS,
                max_tokens=1024,
                frequency_penalty=0.5,
                presence_penalty=0.2,
            )
        resp = await _retry_api_call(make_call, label=label)
        if resp is None:
            return ""
        try:
            answer = resp.choices[0].message.content.strip()
            del resp
            return answer
        except (IndexError, AttributeError):
            return ""

async def generate_answer(question: str, chunk: str, voice_mode: str, idx: int = 0) -> str:
    """Generate an answer in Shakespeare's voice for the given voice mode.

    Retries once OUTSIDE the semaphore if template detected.
    """
    system_prompt = make_generation_prompt(voice_mode)

    # Get voice-mode-specific opener cues and randomly select 3 for variety
    mode_data = VOICE_MODES.get(voice_mode, {})
    opener_cues = mode_data.get("opener_cues", [])

    if opener_cues:
        selected = random.sample(opener_cues, min(3, len(opener_cues)))
        cue_lines = "\n".join(f"  • {c}" for c in selected)
        opener_instruction = (
            f"For THIS specific response, try one of these opening approaches:\n"
            f"{cue_lines}\n"
            f"Pick whichever fits the question best. Do NOT reuse the same opening "
            f"pattern you used in previous answers."
        )
    else:
        opener_instruction = (
            "Start with something only YOU would say in this voice mode — a vivid image from your plays, "
            "a characteristic phrase, a direct answer, a rhetorical question in your style, "
            "or a reference to a specific scene you wrote."
        )

    user_prompt = (
        f"Use the following passage from one of your plays to inform your answer, but respond naturally "
        f"as yourself — do not quote it directly or reference 'a text':\n"
        f"---\n{chunk}\n---\n\n"
        f"Question: {question}\n\n"
        f"CRITICAL: Your opening sentence must be completely unique and specific to this answer. "
        f"Do NOT begin with any of these: 'The weight of', 'My friend', 'The memory', "
        f"'The memories', 'My child', 'I remember', 'I recall', 'You see', 'Ah', "
        f"'Brother', 'Friend', 'Let me tell you', 'Well', 'Indeed', 'In truth', "
        f"'It is a question'.\n\n"
        f"{opener_instruction}"
    )

    # Attempt 1 — release semaphore fully before any retry
    answer = await _single_answer_call(system_prompt, user_prompt, label=f"A-{idx}")

    # Retry once if template detected (semaphore was released, so no deadlock)
    if answer and is_template_answer(answer):
        answer = await _single_answer_call(system_prompt, user_prompt, label=f"A-{idx}-tmpl")

    return answer


# ============================================================================
# CHECKPOINT HELPERS
# ============================================================================
def _partial_path(ckpt_dir: str, label: str, round_idx: int) -> str:
    """Path for a per-round partial file: e.g. _checkpoints/hamlet.r0.partial.jsonl"""
    return f"{ckpt_dir}/{label}.r{round_idx}.partial.jsonl"

def _done_path(ckpt_dir: str, label: str, round_idx: int) -> str:
    """Path for round completion marker: e.g. _checkpoints/hamlet.r0.done"""
    return f"{ckpt_dir}/{label}.r{round_idx}.done"

def _count_lines(path: str) -> int:
    if not os.path.exists(path):
        return 0
    with open(path) as f:
        return sum(1 for _ in f)

def _mark_round_done(ckpt_dir: str, label: str, round_idx: int):
    """Create a marker file indicating a round has been fully processed."""
    Path(_done_path(ckpt_dir, label, round_idx)).touch()

def _is_round_done(ckpt_dir: str, label: str, round_idx: int) -> bool:
    """Check if a round was fully processed.
    A round is done if it has a .done marker OR a non-empty partial file.
    """
    if os.path.exists(_done_path(ckpt_dir, label, round_idx)):
        return True
    pf = _partial_path(ckpt_dir, label, round_idx)
    return os.path.exists(pf) and _count_lines(pf) > 0

def _merge_partials(ckpt_dir: str, label: str, outfile: str, num_rounds: int):
    """Merge per-round partial files into the final source .jsonl, then delete partials and markers."""
    with open(outfile, "w") as out:
        for r in range(num_rounds):
            pf = _partial_path(ckpt_dir, label, r)
            if os.path.exists(pf):
                with open(pf) as inp:
                    for line in inp:
                        out.write(line)
                os.remove(pf)
            done = _done_path(ckpt_dir, label, r)
            if os.path.exists(done):
                os.remove(done)


# ── INTER-SOURCE COOLDOWN ──
SOURCE_COOLDOWN = 15        # seconds between sources to let rate limits recover

# ── Process ONE SOURCE FILE AT A TIME: read → chunk → Q → A → write per round → merge ──
qa_dir = f"{OUTPUT_DIR}/per_source"
os.makedirs(qa_dir, exist_ok=True)
ckpt_dir = f"{qa_dir}/_checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
grand_total = 0
template_reject_total = 0

for source_idx, entry in enumerate(source_registry):
    filepath = entry["filepath"]
    voice_mode = entry["voice_mode"]
    label = entry["label"]
    outfile = f"{qa_dir}/{label}.jsonl"

    # ── Resume check: final file already exists with content? ACCEPT IT. ──
    if os.path.exists(outfile):
        existing = _count_lines(outfile)
        if existing > 0:
            print(f"  {label:20s} [{voice_mode:12s}] SKIP ({existing} QA pairs — already generated)")
            grand_total += existing
            continue
        else:
            print(f"  {label:20s} [{voice_mode:12s}] EMPTY final file — removing, will check partials")
            os.remove(outfile)

    # ── Figure out which rounds are already done (partial files / done markers) ──
    rounds_done = {}
    for r in range(NUM_ROUNDS):
        if _is_round_done(ckpt_dir, label, r):
            pf = _partial_path(ckpt_dir, label, r)
            count = _count_lines(pf) if os.path.exists(pf) else 0
            rounds_done[r] = count

    if len(rounds_done) == NUM_ROUNDS:
        total_partial = sum(rounds_done.values())
        print(f"  {label:20s} [{voice_mode:12s}] All rounds done in partials ({total_partial} QA) — merging")
        _merge_partials(ckpt_dir, label, outfile, NUM_ROUNDS)
        grand_total += total_partial
        continue

    # ── Inter-source cooldown (skip for the first source) ──
    if source_idx > 0:
        print(f"\n  ⏳ Cooling down {SOURCE_COOLDOWN}s before next source (rate limit recovery)...")
        await asyncio.sleep(SOURCE_COOLDOWN)

    # Read and chunk THIS source file only
    with open(filepath) as f:
        text = f.read()
    chunks = chunk_text(text)
    del text

    # Apply test limit if set
    if TEST_CHUNKS_PER_ROUND:
        chunks = chunks[:TEST_CHUNKS_PER_ROUND]

    rounds_to_run = [r for r in range(NUM_ROUNDS) if r not in rounds_done]
    skipped_total = sum(rounds_done.values())

    # Calculate expected values for display (based on actual chunk count)
    expected_qa = len(chunks) * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    expected_per_round = len(chunks) * QUESTIONS_PER_CHUNK

    # Reset error counters for this source
    _error_counts.clear()

    print(f"\n{'='*60}")
    test_tag = f" [TEST MODE: {len(chunks)} chunks]" if TEST_CHUNKS_PER_ROUND else ""
    print(f"  {label.upper()} ({voice_mode}) — {len(chunks)} chunks × {QUESTIONS_PER_CHUNK} Q/chunk{test_tag}")
    print(f"  Rounds to run: {rounds_to_run} (skipping {list(rounds_done.keys())} with {skipped_total} QA on disk)")
    print(f"  Expected total: ~{expected_qa} QA pairs")
    print(f"{'='*60}")

    for round_idx in range(NUM_ROUNDS):
        round_name = ['Factual', 'Craft', 'Reflective'][round_idx % 3]
        pf = _partial_path(ckpt_dir, label, round_idx)

        if round_idx in rounds_done:
            print(f"  {label} R{round_idx+1} ({round_name}) — SKIP ({rounds_done[round_idx]} QA on disk)")
            continue

        # Generate questions — with chunk index for error labels
        q_tasks = [generate_questions_for_chunk(c, round_idx, voice_mode, chunk_idx=i) for i, c in enumerate(chunks)]
        q_results = await atqdm.gather(*q_tasks, desc=f"  {label} R{round_idx+1} ({round_name}) Q")

        qa_batch = []
        total_questions = 0
        empty_chunks = 0
        for chunk, questions in zip(chunks, q_results):
            if not questions:
                empty_chunks += 1
            for q in questions:
                q = q.strip()
                if len(q) > 15:
                    qa_batch.append({"chunk": chunk, "question": q})
                    total_questions += 1

        # ── YIELD DIAGNOSTIC ──
        q_yield_pct = (total_questions / expected_per_round * 100) if expected_per_round else 0
        yield_status = "✓" if q_yield_pct > 80 else "⚠" if q_yield_pct > 40 else "✗"
        print(f"  {yield_status} Q yield: {total_questions}/{expected_per_round} ({q_yield_pct:.0f}%) — "
              f"{empty_chunks}/{len(chunks)} chunks returned 0 questions")

        if total_questions == 0:
            print(f"    ✗ No questions generated — skipping answer phase for this round")
            print(f"    Error breakdown: {dict(_error_counts)}")
            _mark_round_done(ckpt_dir, label, round_idx)
            with open(pf, "w") as f:
                pass
            continue

        del q_tasks, q_results
        gc.collect()

        # ── Small cooldown between Q and A phases to ease rate pressure ──
        if q_yield_pct < 80:
            cooldown = 10
            print(f"    ⏳ Low Q yield — cooling down {cooldown}s before answer phase...")
            await asyncio.sleep(cooldown)

        # Generate answers with template rejection
        a_tasks = [generate_answer(qa["question"], qa["chunk"], voice_mode, idx=i) for i, qa in enumerate(qa_batch)]
        a_results = await atqdm.gather(*a_tasks, desc=f"  {label} R{round_idx+1} ({round_name}) A")
        del a_tasks

        # Write results — filter out template answers AND short answers
        round_count = 0
        round_template_rejects = 0
        round_empty = 0
        with open(pf, "w") as f:
            for qa, answer in zip(qa_batch, a_results):
                if len(answer) < 20:
                    round_empty += 1
                    continue
                if is_template_answer(answer):
                    round_template_rejects += 1
                    continue
                item = {
                    "persona": "shakespeare",
                    "voice_mode": voice_mode,
                    "source_label": label,
                    "question": qa["question"],
                    "answer": answer,
                    "chunk_key": qa["chunk"][:100],
                }
                f.write(json.dumps(item) + "\n")
                round_count += 1

        # Mark round as fully processed
        _mark_round_done(ckpt_dir, label, round_idx)

        template_reject_total += round_template_rejects
        a_yield_pct = (round_count / total_questions * 100) if total_questions else 0
        print(f"  ✓ {label} R{round_idx+1}: {round_count}/{total_questions} QA saved "
              f"(A yield: {a_yield_pct:.0f}%, {round_empty} empty, {round_template_rejects} template) → {pf}")
        del qa_batch, a_results
        gc.collect()

        # Inter-round cooldown
        if round_idx < NUM_ROUNDS - 1:
            await asyncio.sleep(5)

    # All rounds done — merge partials into final file
    _merge_partials(ckpt_dir, label, outfile, NUM_ROUNDS)
    count = _count_lines(outfile)
    grand_total += count
    print(f"  ✓ {label}: {count}/{expected_qa} QA pairs merged → {outfile}")
    del chunks
    gc.collect()
    print(f"  🧹 Memory cleared for {label}")

    # ── Source-level error summary ──
    if any(_error_counts.values()):
        print(f"  📊 Errors this source: {dict(_error_counts)}")

print(f"\n{'='*60}")
print(f"DONE: {grand_total:,} total QA pairs across {len(source_registry)} source files")
print(f"Template answers rejected: {template_reject_total:,}")
print(f"Per-source files in: {qa_dir}/")


  HAMLET (tragic) — 30 chunks × 5 Q/chunk [TEST MODE: 30 chunks]
  Rounds to run: [0, 1, 2] (skipping [] with 0 QA on disk)
  Expected total: ~450 QA pairs


  hamlet R1 (Factual) Q: 100%|██████████| 30/30 [00:30<00:00,  1.01s/it]


  ✓ Q yield: 145/150 (97%) — 1/30 chunks returned 0 questions


  hamlet R1 (Factual) A: 100%|██████████| 145/145 [03:22<00:00,  1.40s/it]


  ✓ hamlet R1: 145/145 QA saved (A yield: 100%, 0 empty, 0 template) → /home/spark/projects/training/shakespeare/data/training-data/shakespeare_persona/per_source/_checkpoints/hamlet.r0.partial.jsonl


...

  ✓ romeo_juliet R2: 150/150 QA saved (A yield: 100%, 0 empty, 0 template) → /home/spark/projects/training/shakespeare/data/training-data/shakespeare_persona/per_source/_checkpoints/romeo_juliet.r1.partial.jsonl


  romeo_juliet R3 (Reflective) Q: 100%|██████████| 30/30 [00:46<00:00,  1.55s/it]


  ✓ Q yield: 150/150 (100%) — 0/30 chunks returned 0 questions


  romeo_juliet R3 (Reflective) A: 100%|██████████| 150/150 [03:25<00:00,  1.37s/it]


  ✓ romeo_juliet R3: 150/150 QA saved (A yield: 100%, 0 empty, 0 template) → /home/spark/projects/training/shakespeare/data/training-data/shakespeare_persona/per_source/_checkpoints/romeo_juliet.r2.partial.jsonl
  ✓ romeo_juliet: 450/450 QA pairs merged → /home/spark/projects/training/shakespeare/data/training-data/shakespeare_persona/per_source/romeo_juliet.jsonl
  🧹 Memory cleared for romeo_juliet
  📊 Errors this source: {'timeout': 1}

DONE: 4,930 total QA pairs across 11 source files
Template answers rejected: 0
Per-source files in: /home/spark/projects/training/shakespeare/data/training-data/shakespeare_persona/per_source/


## 5b. Quality Gate

**Run BEFORE assembly.** Measures template contamination and per-voice-mode opener uniqueness. If template contamination exceeds 15%, the data is NOT safe to assemble — Shakespeare's four registers will sound indistinguishable.

In [7]:
from collections import Counter

qa_dir = f"{OUTPUT_DIR}/per_source"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))

# Analyze template contamination and voice uniqueness per voice mode
print("VOICE MODE DIFFERENTIATION QUALITY GATE\n")
print(f"{'='*70}")

all_openers = {}  # voice_mode -> list of 4-gram openers
global_opener_counts = Counter()
mode_stats = {}

for qa_file in qa_files:
    label = Path(qa_file).stem
    with open(qa_file) as f:
        for line in f:
            item = json.loads(line)
            voice_mode = item.get("voice_mode", "unknown")
            answer = item["answer"].strip()

            if voice_mode not in mode_stats:
                mode_stats[voice_mode] = {"total": 0, "template_count": 0}
            if voice_mode not in all_openers:
                all_openers[voice_mode] = []

            mode_stats[voice_mode]["total"] += 1

            # Check template contamination
            if is_template_answer(answer):
                mode_stats[voice_mode]["template_count"] += 1

            # Track 4-gram opener
            words = answer.split()[:4]
            opener = ' '.join(words)
            all_openers[voice_mode].append(opener)
            global_opener_counts[opener] += 1

# Report per voice mode
print(f"\n{'Voice Mode':<15} {'Total':>6} {'Template':>8} {'Contam%':>8}  Status")
print("-" * 60)
total_all = 0
template_all = 0
for vm, stats in sorted(mode_stats.items(), key=lambda x: -x[1].get("template_count", 0) / max(x[1].get("total", 1), 1)):
    total_all += stats["total"]
    template_all += stats["template_count"]
    contamination_pct = (stats["template_count"] / stats["total"] * 100) if stats["total"] else 0
    status = "✓ PASS" if contamination_pct < 15 else "✗ FAIL" if contamination_pct > 30 else "⚠ WARN"
    print(f"  {vm:<15} {stats['total']:>5} {stats['template_count']:>8} {contamination_pct:>7.1f}%  {status}")

global_contam = (template_all / total_all * 100) if total_all else 0
print(f"\n  {'GLOBAL':<15} {total_all:>5} {template_all:>8} {global_contam:>7.1f}%")

# Cross-mode uniqueness: how many 4-gram openers are shared across voice modes?
print(f"\n{'='*70}")
print(f"CROSS-MODE OPENER ANALYSIS\n")

# Top shared openers
print("Top 10 most repeated opening 4-grams:")
for phrase, count in global_opener_counts.most_common(10):
    who = [vm for vm, ops in all_openers.items() if phrase in ops]
    n_modes = len(who)
    print(f"  {count:>4}x across {n_modes:>2} modes: \"{phrase}\"")

# Per-mode uniqueness
print(f"\nPer-mode opener uniqueness:")
for vm, ops in sorted(all_openers.items()):
    unique_to_mode = sum(1 for o in ops if global_opener_counts[o] == 1)
    pct = unique_to_mode / len(ops) * 100 if ops else 0
    print(f"  {vm:<15} {pct:>5.0f}% unique openers ({unique_to_mode}/{len(ops)})")

# GATE DECISION
print(f"\n{'='*70}")
if global_contam > 30:
    print("✗ QUALITY GATE FAILED — template contamination too high ({:.1f}%)".format(global_contam))
    print("  The generated data will produce homogeneous voices across modes. Do NOT proceed to assembly.")
    print("  Fix: Delete per_source/*.jsonl files for failed sources and re-run generation.")
    QUALITY_GATE_PASSED = False
elif global_contam > 15:
    print("⚠ QUALITY GATE WARNING — template contamination elevated ({:.1f}%)".format(global_contam))
    print("  Consider re-generating the worst offenders. Proceed with caution.")
    QUALITY_GATE_PASSED = True
else:
    print("✓ QUALITY GATE PASSED — template contamination {:.1f}% (target: <15%)".format(global_contam))
    QUALITY_GATE_PASSED = True

del all_openers, global_opener_counts, mode_stats

VOICE MODE DIFFERENTIATION QUALITY GATE


Voice Mode       Total Template  Contam%  Status
------------------------------------------------------------
  tragic           1795        0     0.0%  ✓ PASS
  historical       1340        0     0.0%  ✓ PASS
  comic            1345        0     0.0%  ✓ PASS
  romantic          450        0     0.0%  ✓ PASS

  GLOBAL           4930        0     0.0%

CROSS-MODE OPENER ANALYSIS

Top 10 most repeated opening 4-grams:
   337x across  1 modes: "A crown is not"
    96x across  1 modes: "What is it that"
    54x across  1 modes: "Once more unto the"
    46x across  1 modes: "O, how the stars"
    32x across  1 modes: "A crown is a"
    29x across  3 modes: "A man may wear"
    28x across  2 modes: "A crown sits heavy"
    28x across  2 modes: "Mark me, when a"
    28x across  1 modes: "Mark you how the"
    27x across  1 modes: "What is it in"

Per-mode opener uniqueness:
  comic              56% unique openers (748/1345)
  historical         36% un

## 6. Assemble Conversations & Save

Read per-source QA files from disk one at a time, group into multi-turn conversations, quality-filter, and write final ShareGPT JSONL.

**ONE system prompt for ALL conversations** — the unified Shakespeare prompt from `prompts/shakespeare.md`. The model learns to shift registers based on what's being discussed, not based on which system prompt it receives. Voice mode metadata is preserved for analytics but is NOT part of the training signal.

**Only proceed if the Quality Gate above passed.**

In [8]:
# Check quality gate before proceeding
if not QUALITY_GATE_PASSED:
    raise RuntimeError(
        "Quality gate FAILED. Template contamination too high. "
        "Delete bad per_source/*.jsonl files and re-run generation before assembling."
    )

def quality_check(conv):
    """Reject AI-speak AND template answers."""
    for msg in conv["conversations"]:
        if msg["from"] == "gpt":
            v = msg["value"]
            if len(v) < 30:
                return False
            lower = v.lower()
            # Reject AI refusals
            if any(p in lower for p in ["as an ai", "as a language model", "i cannot fulfill", "i'm sorry, but"]):
                return False
            # Reject template openers (belt-and-suspenders with generation-time filter)
            if is_template_answer(v):
                return False
    return True

total_convs = 0
template_filtered_convs = 0
qa_dir = f"{OUTPUT_DIR}/per_source"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))
print(f"Reading {len(qa_files)} per-source files\n")

# Write conversations streaming — never hold all sources in memory at once
with open(OUTPUT_FILE, "w") as out_f:
    for qa_file in qa_files:
        label = Path(qa_file).stem

        # Read this source's QA pairs
        items = []
        with open(qa_file) as f:
            for line in f:
                items.append(json.loads(line))

        if not items:
            print(f"  {label:20s}    0 QA →    0 conversations (empty)")
            continue

        # Determine voice_mode from the first item (all items in a file share the same mode)
        voice_mode = items[0].get("voice_mode", "tragic")

        # Group by chunk_key (related QAs stay together)
        groups = defaultdict(list)
        for item in items:
            groups[item["chunk_key"]].append(item)

        # Build multi-turn conversations with voice-mode-specific system prompt
        source_count = 0
        for _, group_items in groups.items():
            random.shuffle(group_items)
            for i in range(0, len(group_items), TURNS_PER_CONVERSATION):
                batch = group_items[i:i + TURNS_PER_CONVERSATION]
                if len(batch) < 2:
                    continue
                conv = {
                    "voice_mode": voice_mode,  # metadata for analytics only
                    "conversations": [
                        {"from": "system", "value": UNIFIED_SYSTEM_PROMPT}
                    ],
                }
                for qa in batch:
                    conv["conversations"].append({"from": "human", "value": qa["question"]})
                    conv["conversations"].append({"from": "gpt", "value": qa["answer"]})
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    source_count += 1
                else:
                    template_filtered_convs += 1

        total_convs += source_count
        print(f"  {label:20s} [{voice_mode:12s}] {len(items):>5} QA → {source_count:>4} conversations")
        del items, groups
        gc.collect()

print(f"\n✓ Saved {total_convs:,} conversations to:")
print(f"  {OUTPUT_FILE}")
print(f"  ({os.path.getsize(OUTPUT_FILE) / 1024 / 1024:.1f} MB)")
if template_filtered_convs:
    print(f"  (filtered {template_filtered_convs} conversations with template answers)")

# Shuffle the output file for better training
import subprocess
subprocess.run(["shuf", OUTPUT_FILE, "-o", OUTPUT_FILE])
print(f"  ✓ Shuffled output file")

Reading 11 per-source files

  hamlet               [tragic      ]   445 QA →  119 conversations
  henry_v              [historical  ]   440 QA →  117 conversations
  julius_caesar        [historical  ]   450 QA →  120 conversations
  king_lear            [tragic      ]   450 QA →  120 conversations
  macbeth              [tragic      ]   450 QA →  120 conversations
  midsummer            [comic       ]   450 QA →  120 conversations
  much_ado             [comic       ]   450 QA →  120 conversations
  othello              [tragic      ]   450 QA →  120 conversations
  richard_iii          [historical  ]   450 QA →  120 conversations
  romeo_juliet         [romantic    ]   450 QA →  120 conversations
  twelfth_night        [comic       ]   445 QA →  119 conversations

✓ Saved 1,315 conversations to:
  /home/spark/projects/training/shakespeare/data/training-data/shakespeare_persona/shakespeare_sharegpt.jsonl
  (22.1 MB)
  ✓ Shuffled output file


## 7. Verify

In [9]:
# Verify by streaming from disk — no need to hold all conversations in RAM
mode_dist = defaultdict(int)
total_turns = 0
total_convs_verify = 0
sample_convs = []

with open(OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        conv = json.loads(line)
        total_convs_verify += 1
        total_turns += len(conv["conversations"]) - 1

        # Voice mode stored as metadata field (not in system prompt)
        detected_mode = conv.get("voice_mode", "unknown")
        mode_dist[detected_mode] += 1

        # Reservoir sample: keep up to 3 random conversations (one per mode ideally)
        if len(sample_convs) < 4:
            sample_convs.append(conv)
        else:
            j = random.randint(0, line_num)
            if j < 4:
                sample_convs[j] = conv

        del conv

print("VOICE MODE DISTRIBUTION:")
print(f"{'Voice Mode':20s} {'Convs':>6s} {'%':>6s}")
print("-" * 35)
for vm, c in sorted(mode_dist.items(), key=lambda x: -x[1]):
    print(f"  {vm:20s} {c:>5d}  {c/total_convs_verify*100:>5.1f}%")

print(f"\n{'='*50}")
print(f"TOTAL: {total_convs_verify:,} conversations, {total_turns:,} turns ({total_turns//2:,} QA pairs)")
print(f"Persona: shakespeare (single persona, {len(VOICE_MODES)} voice modes)")

# Sample conversations
print(f"\n{'='*50}")
print("SAMPLE CONVERSATIONS:\n")
for conv in sample_convs:
    print(f"{'─'*50}")
    for msg in conv["conversations"]:
        role = msg["from"].upper()
        text = msg["value"][:250] + ("..." if len(msg["value"]) > 250 else "")
        print(f"[{role}] {text}\n")

del sample_convs
gc.collect()

print(f"\n✓ Data ready for training. Load this file in your training notebook:")
print(f"  {OUTPUT_FILE}")

VOICE MODE DISTRIBUTION:
Voice Mode            Convs      %
-----------------------------------
  tragic                 479   36.4%
  comic                  359   27.3%
  historical             357   27.1%
  romantic               120    9.1%

TOTAL: 1,315 conversations, 9,858 turns (4,929 QA pairs)
Persona: shakespeare (single persona, 4 voice modes)

SAMPLE CONVERSATIONS:

──────────────────────────────────────────────────
[SYSTEM] ### **I AM SHAKESPEARE — PLAYER, POET, PLAYWRIGHT**

#### MY EARLY DAYS AND CALLING
I am William Shakespeare — born in Stratford-upon-Avon in the year 1564, son of John Shakespeare, a glover who wore the alderman's gown before debt stripped it from h...

[HUMAN] Looking back on this scene, what truth about human nature did it teach you when a man is made to doubt his own judgment in favor of another's flattery?

[GPT] Mark well how the serpent flatters not to charm, but to unseat the throne within a man’s own breast.  

When a noble mind—say, one born to 

## 8. Augmented Training Data — Continuation Chunks

**Purpose:** Teach the model Shakespeare's language patterns, verse cadence, and dramatic style by exposing it to raw play text as continuation tasks — no API calls needed.

Each play's source text is chunked into ~500-token blocks and formatted as ShareGPT conversations:
- **System:** Unified Shakespeare system prompt (same as Q/A generation)
- **Human:** A brief instruction + a seed (first ~60 tokens of the chunk)
- **GPT:** The remaining ~440 tokens (what the model should learn to produce)

Voice mode metadata is preserved on each continuation entry so the model sees all four registers naturally.

This teaches the model to "think" and "speak" in Shakespeare's distinctive voice — iambic pentameter rhythms, Elizabethan vocabulary, dramatic imagery, etc. — without needing expensive LLM generation.

**No API calls.** Just text extraction and formatting.

### Future augmentation (not yet implemented):
- **Chain-of-Thought (CoT):** Step-by-step dramatic analysis per voice mode (set `ENABLE_COT = True`)
- **Instruction + Response:** Varied tasks beyond Q/A (summarize, explain, compare, compose)
- **Preference / DPO:** Good vs. bad interpretation pairs

In [ ]:
# ============================================================================
# 8a. CONFIGURATION — Augmented Data Generation
# ============================================================================

# ── Feature flags ──
ENABLE_CONTINUATION = True   # Free: chunk source texts into continuation tasks
ENABLE_COT = False           # TODO: Chain-of-thought reasoning (requires API calls)
ENABLE_INSTRUCTIONS = False  # TODO: Varied instruction tasks (requires API calls)

# ── CoT settings (for future use) ──
COT_ENABLE_THINKING = False  # Set True to use <think> tags with Qwen3 thinking model
COT_PASSAGES_PER_VOICE_MODE = 25  # How many passages per voice mode for CoT

# ── Continuation settings ──
CONTINUATION_CHUNK_TOKENS = 500     # Target tokens per chunk
CONTINUATION_SEED_TOKENS = 60       # Tokens given as "seed" in the human turn
CONTINUATION_MIN_COMPLETION = 100   # Min characters in completion portion (skip tiny tails)

# ── Output paths ──
AUGMENTED_DIR = f"{OUTPUT_DIR}/augmented"
CONTINUATION_DIR = f"{AUGMENTED_DIR}/continuation"
COMBINED_OUTPUT_FILE = f"{OUTPUT_DIR}/shakespeare_combined_sharegpt.jsonl"

os.makedirs(CONTINUATION_DIR, exist_ok=True)

# ── Continuation instruction templates (randomly sampled per chunk) ──
CONTINUATION_INSTRUCTIONS = [
    "Continue writing in this voice and style, carrying forward the themes and language:",
    "Continue this passage, maintaining the same tone, vocabulary, and cadence:",
    "Write what comes next, staying true to the voice and spirit of this text:",
    "Carry on from where this passage leaves off, preserving the distinctive style:",
    "Continue this text naturally, as if you were the original playwright:",
]

print("✓ Augmentation config loaded")
print(f"  Continuation: {'ENABLED' if ENABLE_CONTINUATION else 'DISABLED'}")
print(f"  CoT:          {'ENABLED' if ENABLE_COT else 'DISABLED (future)'}")
print(f"  Instructions: {'ENABLED' if ENABLE_INSTRUCTIONS else 'DISABLED (future)'}")
print(f"  CoT thinking: {'ENABLED' if COT_ENABLE_THINKING else 'DISABLED'}")
print(f"  Output:       {COMBINED_OUTPUT_FILE}")

In [ ]:
# ============================================================================
# 8b. GENERATE CONTINUATION CHUNKS — No API calls needed
# ============================================================================
# Reads each play's source text, chunks into ~500-token blocks,
# splits each chunk into seed (human) + completion (gpt), and formats
# as ShareGPT conversations with the unified Shakespeare system prompt.
#
# Key difference from biblical: Shakespeare has ONE persona with 4 voice modes.
# Each continuation entry is tagged with the play's voice_mode metadata.
# ============================================================================

import gc
import tiktoken

# Use cl100k_base as a reasonable approximation for token counting
# (Qwen uses a different tokenizer but this is close enough for chunking)
try:
    _tokenizer = tiktoken.get_encoding("cl100k_base")
except Exception:
    # Fallback: approximate 1 token ≈ 4 chars
    _tokenizer = None
    print("⚠ tiktoken not available, using character-based approximation")


def chunk_text_by_tokens(text: str, max_tokens: int = CONTINUATION_CHUNK_TOKENS) -> list[str]:
    """Split text into chunks of approximately max_tokens tokens.
    Tries to break at sentence boundaries for cleaner chunks."""
    if _tokenizer is None:
        # Fallback: ~4 chars per token
        char_limit = max_tokens * 4
        return chunk_text(text, chunk_size=char_limit, overlap=0)

    tokens = _tokenizer.encode(text)
    chunks = []
    start = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_str = _tokenizer.decode(tokens[start:end])

        # Try to break at a sentence boundary (within last 20% of chunk)
        if end < len(tokens):
            boundary_zone = chunk_str[int(len(chunk_str) * 0.8):]
            for sep in ['. ', '? ', '! ', '.\n', '\n\n']:
                last_break = boundary_zone.rfind(sep)
                if last_break >= 0:
                    actual_end = int(len(chunk_str) * 0.8) + last_break + len(sep)
                    chunk_str = chunk_str[:actual_end]
                    # Recalculate token position for next start
                    end = start + len(_tokenizer.encode(chunk_str))
                    break

        chunk_str = chunk_str.strip()
        if len(chunk_str) > 50:  # Skip tiny fragments
            chunks.append(chunk_str)

        if end >= len(tokens):
            break
        start = end

    return chunks


def split_seed_completion(chunk: str, seed_tokens: int = CONTINUATION_SEED_TOKENS) -> tuple[str, str]:
    """Split a chunk into seed (first ~seed_tokens tokens) and completion (rest).
    Tries to break at a sentence/phrase boundary for natural splits."""
    if _tokenizer is None:
        char_limit = seed_tokens * 4
        # Try to break at a sentence boundary
        seed_zone = chunk[:int(char_limit * 1.3)]
        for sep in ['. ', '? ', '! ', '; ', ', ']:
            idx = seed_zone.rfind(sep)
            if idx > char_limit * 0.5:
                return chunk[:idx + len(sep)].strip(), chunk[idx + len(sep):].strip()
        return chunk[:char_limit].strip(), chunk[char_limit:].strip()

    tokens = _tokenizer.encode(chunk)
    if len(tokens) <= seed_tokens + 20:
        # Chunk too short to split meaningfully
        return None, None

    seed_str = _tokenizer.decode(tokens[:seed_tokens])

    # Try to break at a natural boundary near the seed end
    extended_seed = _tokenizer.decode(tokens[:int(seed_tokens * 1.3)])
    for sep in ['. ', '? ', '! ', '; ', ', ', ' ']:
        idx = extended_seed.rfind(sep)
        if idx > len(seed_str) * 0.6:
            seed_str = extended_seed[:idx + len(sep)].strip()
            break

    completion_str = chunk[len(seed_str):].strip()
    return seed_str, completion_str


if ENABLE_CONTINUATION:
    # Load unified system prompt (same as used for Q/A generation)
    with open(PROMPT_FILE) as f:
        UNIFIED_SYSTEM_PROMPT = f.read().strip()

    # Build a mapping from source filename to voice_mode using PLAY_VOICE_MODES
    # (defined in cell 3 — source discovery). Each source file has a tagged voice mode.
    source_voice_map = {}
    for entry in source_registry:
        source_voice_map[entry["filepath"]] = entry["voice_mode"]

    print("Generating continuation chunks from source texts...\n")

    source_files = sorted(glob.glob(f"{SOURCE_CLEAN_DIR}/*.txt"))
    total_continuations = 0
    persona_chunk_counts = {}

    for filepath in source_files:
        label = Path(filepath).stem
        voice_mode = source_voice_map.get(filepath, "tragic")  # fallback
        outfile = f"{CONTINUATION_DIR}/{label}_continuation.jsonl"

        # Resume: skip if already generated
        if os.path.exists(outfile) and os.path.getsize(outfile) > 0:
            existing = sum(1 for _ in open(outfile))
            print(f"  {label:30s} [{voice_mode:12s}] SKIP ({existing} continuations already on disk)")
            total_continuations += existing
            persona_chunk_counts[label] = existing
            continue

        # Read source text
        with open(filepath) as f:
            text = f.read()

        # Chunk by tokens
        chunks = chunk_text_by_tokens(text, max_tokens=CONTINUATION_CHUNK_TOKENS)
        del text

        # Generate continuation conversations
        play_count = 0
        with open(outfile, "w") as out_f:
            for chunk in chunks:
                seed, completion = split_seed_completion(chunk, seed_tokens=CONTINUATION_SEED_TOKENS)

                if seed is None or completion is None:
                    continue
                if len(completion) < CONTINUATION_MIN_COMPLETION:
                    continue

                # Pick a random instruction
                instruction = random.choice(CONTINUATION_INSTRUCTIONS)

                conv = {
                    "voice_mode": voice_mode,  # Preserve voice mode metadata
                    "conversations": [
                        {"from": "system", "value": UNIFIED_SYSTEM_PROMPT},
                        {"from": "human", "value": f"{instruction}\n\n\"{seed}\""},
                        {"from": "gpt", "value": completion},
                    ],
                    "data_type": "continuation",  # Tag for tracking
                }

                out_f.write(json.dumps(conv) + "\n")
                play_count += 1

        total_continuations += play_count
        persona_chunk_counts[label] = play_count
        print(f"  {label:30s} [{voice_mode:12s}] {len(chunks):>4} chunks → {play_count:>4} continuations → {outfile}")
        del chunks
        gc.collect()

    print(f"\n✓ Generated {total_continuations:,} continuation entries across {len(source_files)} plays")
    print(f"  Files in: {CONTINUATION_DIR}/")
else:
    print("⏭ Continuation generation DISABLED")

## 9. Merge & Assemble Combined Training Data

Merges the original Q/A ShareGPT data with augmented data (continuation chunks, and eventually CoT/instruction).

**Target blend:** ~60% Q/A, ~40% continuation (adjust when CoT/instruction are added later).

Outputs a single shuffled JSONL file ready for Unsloth training.

In [ ]:
# ============================================================================
# 9. MERGE ALL DATA SOURCES INTO COMBINED TRAINING FILE
# ============================================================================
# Combines:
#   1. Original Q/A ShareGPT data (from Section 6)
#   2. Continuation chunks (from Section 8b)
#   3. [Future] CoT data
#   4. [Future] Instruction data
#
# Tags each entry with "data_type" for tracking blend ratios.
# Applies target blend ratios via upsampling/downsampling.
# ============================================================================

import subprocess
import gc

# ── Target blend (adjust as you add more data types) ──
TARGET_BLEND = {
    "qa": 0.60,
    "continuation": 0.40,
    # "cot": 0.20,         # Uncomment when CoT is implemented
    # "instruction": 0.10,  # Uncomment when instruction tasks are implemented
}

# ── Collect data from all sources ──
data_pools = defaultdict(list)

# 1. Original Q/A conversations
qa_source = OUTPUT_FILE  # shakespeare_sharegpt.jsonl
if os.path.exists(qa_source):
    with open(qa_source) as f:
        for line in f:
            entry = json.loads(line)
            entry["data_type"] = "qa"
            data_pools["qa"].append(entry)
    print(f"  Q/A:           {len(data_pools['qa']):>6,} conversations from {qa_source}")
else:
    print(f"  ⚠ Q/A file not found: {qa_source}")

# 2. Continuation chunks
if ENABLE_CONTINUATION:
    cont_files = sorted(glob.glob(f"{CONTINUATION_DIR}/*_continuation.jsonl"))
    for cf in cont_files:
        with open(cf) as f:
            for line in f:
                entry = json.loads(line)
                entry.setdefault("data_type", "continuation")
                data_pools["continuation"].append(entry)
    print(f"  Continuation:  {len(data_pools['continuation']):>6,} entries from {len(cont_files)} files")

# 3. [Future] CoT
# if ENABLE_COT:
#     cot_files = sorted(glob.glob(f"{AUGMENTED_DIR}/cot/*_cot.jsonl"))
#     for cf in cot_files:
#         with open(cf) as f:
#             for line in f:
#                 entry = json.loads(line)
#                 entry.setdefault("data_type", "cot")
#                 data_pools["cot"].append(entry)
#     print(f"  CoT:           {len(data_pools['cot']):>6,} entries")

# 4. [Future] Instruction
# if ENABLE_INSTRUCTIONS:
#     inst_files = sorted(glob.glob(f"{AUGMENTED_DIR}/instruction/*_instruction.jsonl"))
#     ...

# ── Calculate blend sizes ──
# Use Q/A as anchor: Q/A count = 60% of total → total = Q/A count / 0.6
qa_count = len(data_pools.get("qa", []))
if qa_count == 0:
    raise RuntimeError("No Q/A data found. Run sections 5-6 first.")

# Calculate how many of each type we need
active_types = {k: v for k, v in TARGET_BLEND.items() if k in data_pools and len(data_pools[k]) > 0}
total_target = int(qa_count / active_types.get("qa", 0.6))

print(f"\n  Target total:  {total_target:>6,} conversations")
print(f"  Active blend:  {active_types}")

# ── Sample/repeat each pool to hit target counts ──
combined = []
for dtype, fraction in active_types.items():
    pool = data_pools[dtype]
    target_count = int(total_target * fraction)
    
    if len(pool) >= target_count:
        # Downsample: randomly select target_count
        sampled = random.sample(pool, target_count)
        action = "downsampled"
    else:
        # Upsample: repeat pool + sample remainder
        repeats = target_count // len(pool)
        remainder = target_count % len(pool)
        sampled = pool * repeats + random.sample(pool, remainder)
        action = f"upsampled ({repeats}x + {remainder})"
    
    combined.extend(sampled)
    print(f"  {dtype:15s} {len(pool):>6,} available → {len(sampled):>6,} selected ({action})")

# ── Shuffle ──
random.shuffle(combined)

# ── Write combined file ──
with open(COMBINED_OUTPUT_FILE, "w") as f:
    for entry in combined:
        f.write(json.dumps(entry) + "\n")

file_size_mb = os.path.getsize(COMBINED_OUTPUT_FILE) / 1024 / 1024

print(f"\n✓ Combined training file saved:")
print(f"  {COMBINED_OUTPUT_FILE}")
print(f"  {len(combined):,} conversations ({file_size_mb:.1f} MB)")

# Also shuffle with shuf for extra randomness
subprocess.run(["shuf", COMBINED_OUTPUT_FILE, "-o", COMBINED_OUTPUT_FILE])
print(f"  ✓ Shuffled with shuf")

del combined, data_pools
gc.collect()

## 10. Verify Combined Dataset

Quality checks on the merged dataset:
- Data type distribution (Q/A vs continuation vs future types)
- Voice mode distribution across all data types
- Turn count distribution
- Sample conversations from each data type
- Format validation for Unsloth/ShareGPT compatibility

In [ ]:
# ============================================================================
# 10. VERIFY COMBINED DATASET
# ============================================================================

import gc
from collections import Counter

# ── Stream through combined file ──
type_counts = Counter()
mode_by_type = defaultdict(lambda: defaultdict(int))
turn_dist = Counter()
format_errors = []
samples_by_type = {}  # data_type -> sample conversation
total_entries = 0

with open(COMBINED_OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        entry = json.loads(line)
        total_entries += 1
        dtype = entry.get("data_type", "qa")  # Default to qa for old entries
        type_counts[dtype] += 1

        convs = entry.get("conversations", [])
        turn_dist[len(convs)] += 1

        # Format validation
        if len(convs) < 3:
            format_errors.append((line_num, f"Too few turns: {len(convs)}"))
            continue
        if convs[0]["from"] != "system":
            format_errors.append((line_num, f"First turn not system: {convs[0]['from']}"))
            continue
        for j in range(1, len(convs)):
            expected = "human" if j % 2 == 1 else "gpt"
            if convs[j]["from"] != expected:
                format_errors.append((line_num, f"Turn {j} wrong role: {convs[j]['from']} (expected {expected})"))
                break

        # Voice mode detection (stored as metadata field)
        detected_mode = entry.get("voice_mode", "unknown")
        mode_by_type[dtype][detected_mode] += 1

        # Collect one sample per type
        if dtype not in samples_by_type:
            samples_by_type[dtype] = entry

# ── Report ──
print("=" * 70)
print("COMBINED DATASET VERIFICATION")
print("=" * 70)

print(f"\nTotal entries: {total_entries:,}")
print(f"Format errors: {len(format_errors)}")
if format_errors:
    print(f"  First 5 errors:")
    for idx, msg in format_errors[:5]:
        print(f"    Line {idx}: {msg}")

# Data type distribution
print(f"\n{'─' * 50}")
print(f"DATA TYPE DISTRIBUTION")
print(f"{'─' * 50}")
print(f"{'Type':<20} {'Count':>8} {'%':>8}  Bar")
for dtype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    pct = count / total_entries * 100
    bar = "█" * int(pct / 2)
    print(f"  {dtype:<18} {count:>7,} {pct:>7.1f}%  {bar}")
print(f"  {'TOTAL':<18} {total_entries:>7,}")

# Turn distribution
print(f"\n{'─' * 50}")
print(f"TURN COUNT DISTRIBUTION")
print(f"{'─' * 50}")
for turns, count in sorted(turn_dist.items()):
    print(f"  {turns} turns: {count:>6,}")

# Voice mode distribution per data type
print(f"\n{'─' * 50}")
print(f"VOICE MODE DISTRIBUTION BY DATA TYPE")
print(f"{'─' * 50}")
for dtype in sorted(mode_by_type.keys()):
    mode_counts = mode_by_type[dtype]
    total_for_type = sum(mode_counts.values())
    print(f"\n  [{dtype.upper()}] ({total_for_type} total):")
    for vm, c in sorted(mode_counts.items(), key=lambda x: -x[1]):
        bar = "▪" * max(1, c * 30 // max(mode_counts.values()))
        print(f"    {vm:<20} {c:>5}  {bar}")

# Samples
print(f"\n{'─' * 50}")
print(f"SAMPLE CONVERSATIONS (one per data type)")
print(f"{'─' * 50}")
for dtype, sample in samples_by_type.items():
    print(f"\n  ── {dtype.upper()} ──")
    for msg in sample["conversations"]:
        role = msg["from"].upper()
        text = msg["value"][:200] + ("..." if len(msg["value"]) > 200 else "")
        print(f"  [{role}] {text}\n")

print(f"\n{'=' * 70}")
if len(format_errors) == 0:
    print(f"✓ Dataset valid. Ready for training:")
    print(f"  {COMBINED_OUTPUT_FILE}")
    print(f"\n  Load this in your training notebook instead of the Q/A-only file.")
else:
    print(f"⚠ {len(format_errors)} format errors found. Review before training.")

del type_counts, mode_by_type, turn_dist, samples_by_type
gc.collect()